# CLM-002 Gate 0 — CPU only
Enable Internet and existing GITHUB_TOKEN. Run All creates a GitHub results/clm002-gate0-* report branch; it is not read-only. HF_TOKEN optionally reads one pinned parent artifact into /tmp for CPU inspection. No private weights are placed in Save Version Output or Git. No GPU scientific training or Notebook pytest suite.


In [ ]:
from pathlib import Path
import hashlib
import importlib.util
import json
import os
import subprocess
import sys

# Gate 0: CPU engineering only. Run All writes reports and a GitHub results branch; no scientific training.
SOURCE_SHA='3ca3e4433cb82fdcb43066984afb925e6bfc766a'
CODE=Path('/kaggle/working')/('clm002-code-'+SOURCE_SHA[:12])
BASE_REL='research/projects/cellular-language-models/experiments/clm-002-cell-differentiation-joint-value'
OUTPUT=Path('/kaggle/working/clm002-gate0-output')
PATTERNS=['/pyproject.toml', '/src/', '/shared/python/', '/research/projects/cellular-language-models/engineering/kaggle-continuation/', '/research/projects/cellular-language-models/experiments/006-path-first-clm/', '/research/projects/cellular-language-models/experiments/005-compositional-cell-graph/implementation/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/implementation/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r2/implementation/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3/implementation/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m/run.py', '/research/projects/continual-learning/context/04-continual-learning/scripts/research/', '/research/projects/cellular-language-models/context/08-hybrid-clm/scripts/research/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3b/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3c/moe-bridge/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3d/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3e/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3f-connection-admission/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3f-v2/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3g/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3h/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3i/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3j/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3k/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3l/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m/implementation/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m/registration/artifact-recovery-001/independent-review-002/tokenizer.json', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m-e/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m-f/audits/tiny-failure-001/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m-f-d1/', '/research/projects/cellular-language-models/experiments/jct-001/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m/registration/GLOBAL_DATASET_USAGE_LEDGER.json', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m/registration/content-isolation-001/REVIEW.md', '/research/projects/cellular-language-models/experiments/clm-002-cell-differentiation-joint-value/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3e/implementation/minicells_r3e/graph.py', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3e/implementation/minicells_r3e/policy_proposal.py', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3e/implementation/minicells_r3e/stable_connection.py', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3h/config/frozen.json', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3h/implementation/minicells_r3h/routing.py', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3h/implementation/minicells_r3h/trainer.py', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m-e/release/gate2/SESSION_B.json', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m/registration/content-isolation-001/ADDITIONAL_COVERAGE_GAPS.json', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m/registration/content-isolation-001/HISTORICAL_SOURCE_COVERAGE.json', '/research/projects/cellular-language-models/experiments/clm-002-cell-differentiation-joint-value/config/HISTORICAL_REFERENCE_BINDINGS.json', '/research/08-hybrid-clm/experiments/005-compositional-cell-graph/artifacts/7b5affd42efe9d341a9c2cbf73511f67adcedd38/seed-50713001/M0/assets-manifest.json', '/research/08-hybrid-clm/experiments/005-compositional-cell-graph/artifacts/7b5affd42efe9d341a9c2cbf73511f67adcedd38/seed-50713001/P0/assets-manifest.json', '/research/governance/registry.yaml', '/research/projects/cellular-computing/context/03-routing-and-growth/hypothesis/sources/experiment-025-protocol.json', '/research/projects/cellular-computing/experiments/005-consumer-language-bridge/artifacts/corpus-manifest.json', '/research/projects/cellular-computing/experiments/005-consumer-language-bridge/artifacts/model-configs.json', '/research/projects/cellular-computing/experiments/005-consumer-language-bridge/artifacts/task-spec.json', '/research/projects/cellular-computing/experiments/005b-consumer-language-ablation/artifacts/corpus-manifest.json', '/research/projects/cellular-computing/experiments/005b-consumer-language-ablation/artifacts/model-configs.json', '/research/projects/cellular-computing/experiments/006-consumer-language-scaling/artifacts/corpus-manifest.json', '/research/projects/cellular-computing/experiments/006-consumer-language-scaling/artifacts/model-configs.json', '/research/projects/cellular-computing/experiments/007-minicells-30m/artifacts/corpus-manifest.json', '/research/projects/cellular-computing/experiments/007-minicells-30m/artifacts/model-configs.json', '/research/projects/cellular-computing/experiments/009-2d-latent-tissue/artifacts/corpus-manifest.json', '/research/projects/cellular-computing/experiments/011-stabilizing-cost/artifacts/corpus-manifest.json', '/research/projects/cellular-computing/experiments/012-settling-dynamics/artifacts/corpus-manifest.json', '/research/projects/cellular-computing/experiments/013-random-depth-ablation/artifacts/corpus-manifest.json', '/research/projects/cellular-computing/experiments/014-multiseed-core-recipe/artifacts/corpus-manifest.json', '/research/projects/cellular-computing/experiments/016-growing-cellular-lm/artifacts/corpus-manifest.json', '/research/projects/cellular-computing/experiments/016-growing-cellular-lm/artifacts/task-spec.json', '/research/projects/cellular-computing/experiments/017-localized-cellular-learning/artifacts/corpus-manifest.json', '/research/projects/cellular-computing/experiments/017-localized-cellular-learning/artifacts/task-spec.json', '/research/projects/cellular-computing/experiments/018-conditional-tissue-recruitment/artifacts/corpus-manifest.json', '/research/projects/cellular-computing/experiments/018-conditional-tissue-recruitment/artifacts/task-spec.json', '/research/projects/cellular-computing/experiments/018b-feedback-isolated-pressure-recruitment/artifacts/corpus-manifest.json', '/research/projects/cellular-computing/experiments/018b-feedback-isolated-pressure-recruitment/artifacts/task-spec.json', '/research/projects/cellular-computing/experiments/019-proposal-utility-discovery-stable/artifacts/corpus-manifest.json', '/research/projects/cellular-computing/experiments/020-capability-tissue-specificity/artifacts/corpus-manifest.json', '/research/projects/cellular-computing/experiments/021-conflict-driven-differentiation/artifacts/corpus-manifest.json', '/research/projects/cellular-computing/experiments/021-conflict-driven-differentiation/artifacts/task-spec.json', '/research/projects/cellular-computing/experiments/022-emergent-trait-bifurcation/artifacts/corpus-manifest.json', '/research/projects/cellular-computing/experiments/023-online-nonparametric-trait-genesis/artifacts/corpus-manifest.json', '/research/projects/cellular-computing/experiments/023-online-nonparametric-trait-genesis/artifacts/task-spec.json', '/research/projects/cellular-computing/experiments/023b-probationary-trait-genesis/artifacts/corpus-manifest.json', '/research/projects/cellular-computing/experiments/023b-probationary-trait-genesis/artifacts/task-spec.json', '/research/projects/cellular-computing/experiments/024-sequential-probationary-genesis/artifacts/corpus-manifest.json', '/research/projects/cellular-computing/experiments/024b-independent-third-trait/artifacts/corpus-manifest.json', '/research/projects/cellular-computing/experiments/clm-upcycling-study-001-inherit-then-differentiate/artifacts/corpus-manifest.json', '/research/projects/cellular-computing/experiments/clm-v2-validation-001b-closed-loop-handoff/artifacts/corpus-manifest.json', '/research/projects/cellular-computing/experiments/clm-validation-001-program-conditionality/artifacts/corpus-manifest.json', '/research/projects/cellular-computing/experiments/clm-validation-001b-stable-program-conditionality/artifacts/corpus-manifest.json', '/research/projects/cellular-language-models/context/06-native-clm/hypothesis/configs/native-clm-v0-m1-12m.json', '/research/projects/cellular-language-models/experiments/001-global-router/config/registered.yaml', '/research/projects/cellular-language-models/experiments/002-cell-internal-router/artifacts/3c19e9ca5dfc5137716bf0a60d176a5bf6cb261c/seed-42/config.json', '/research/projects/cellular-language-models/experiments/002-cell-internal-router/artifacts/3c19e9ca5dfc5137716bf0a60d176a5bf6cb261c/seed-42/data.json', '/research/projects/cellular-language-models/experiments/002-cell-internal-router/artifacts/3c19e9ca5dfc5137716bf0a60d176a5bf6cb261c/seed-42/protocol.json', '/research/projects/cellular-language-models/experiments/002-cell-internal-router/artifacts/3c19e9ca5dfc5137716bf0a60d176a5bf6cb261c/seed-42/source-config.json', '/research/projects/cellular-language-models/experiments/002-cell-internal-router/artifacts/historical/3c19e9ca5dfc5137716bf0a60d176a5bf6cb261c/seed-42/config.json', '/research/projects/cellular-language-models/experiments/002-cell-internal-router/artifacts/historical/3c19e9ca5dfc5137716bf0a60d176a5bf6cb261c/seed-42/data.json', '/research/projects/cellular-language-models/experiments/002-cell-internal-router/config/registered.yaml', '/research/projects/cellular-language-models/experiments/002-cell-internal-router/validation/routing-002-cell-internal-routing/protocol.json', '/research/projects/cellular-language-models/experiments/003-hard-cell-transition/artifacts/819df6113ca381a79e901330b7e0a1c394e0ee3b/seed-42/config.json', '/research/projects/cellular-language-models/experiments/003-hard-cell-transition/artifacts/819df6113ca381a79e901330b7e0a1c394e0ee3b/seed-42/data.json', '/research/projects/cellular-language-models/experiments/003-hard-cell-transition/artifacts/819df6113ca381a79e901330b7e0a1c394e0ee3b/seed-42/protocol.json', '/research/projects/cellular-language-models/experiments/003-hard-cell-transition/artifacts/819df6113ca381a79e901330b7e0a1c394e0ee3b/seed-42/source-config.yaml', '/research/projects/cellular-language-models/experiments/003-hard-cell-transition/artifacts/historical/819df6113ca381a79e901330b7e0a1c394e0ee3b/seed-42/config.json', '/research/projects/cellular-language-models/experiments/003-hard-cell-transition/artifacts/historical/819df6113ca381a79e901330b7e0a1c394e0ee3b/seed-42/data.json', '/research/projects/cellular-language-models/experiments/003-hard-cell-transition/config/registered.yaml', '/research/projects/cellular-language-models/experiments/003-hard-cell-transition/implementation/runner/config.yaml', '/research/projects/cellular-language-models/experiments/003-hard-cell-transition/implementation/runner/protocol.json', '/research/projects/cellular-language-models/experiments/004-cell-graph/artifacts/6312cefae33c354670d2350b315ac20d1e488a51/seed-42/graph/assets-manifest.json', '/research/projects/cellular-language-models/experiments/004-cell-graph/artifacts/6312cefae33c354670d2350b315ac20d1e488a51/seed-42/graph/config.json', '/research/projects/cellular-language-models/experiments/004-cell-graph/artifacts/6312cefae33c354670d2350b315ac20d1e488a51/seed-42/graph/data.json', '/research/projects/cellular-language-models/experiments/004-cell-graph/artifacts/a59402238b686ff4981786f307d4a3f110698d61/seed-42/reference/assets-manifest.json', '/research/projects/cellular-language-models/experiments/004-cell-graph/artifacts/a59402238b686ff4981786f307d4a3f110698d61/seed-42/reference/config.json', '/research/projects/cellular-language-models/experiments/004-cell-graph/artifacts/a59402238b686ff4981786f307d4a3f110698d61/seed-42/reference/data.json', '/research/projects/cellular-language-models/experiments/004-cell-graph/config/registered.yaml', '/research/projects/cellular-language-models/experiments/004-cell-graph/experiment.yaml', '/research/projects/cellular-language-models/experiments/005-compositional-cell-graph/config/proposed.json', '/research/projects/cellular-language-models/experiments/005-compositional-cell-graph/config/registered.json', '/research/projects/cellular-language-models/experiments/005-compositional-cell-graph/experiment.yaml', '/research/projects/cellular-language-models/experiments/005-compositional-cell-graph/registration/assets-manifest.json', '/research/projects/cellular-language-models/experiments/005-compositional-cell-graph/registration/data.json', '/research/projects/cellular-language-models/experiments/006-path-first-clm/config/proposed-61m.json', '/research/projects/cellular-language-models/experiments/006-path-first-clm/config/proposed.json', '/research/projects/cellular-language-models/experiments/006-path-first-clm/extension-parameter-depth/config/proposed.json', '/research/projects/cellular-language-models/experiments/006-path-first-clm/extension-parameter-depth/exploration-fast-001/config/proposed.json', '/research/projects/cellular-language-models/experiments/006-path-first-clm/extension-parameter-depth/exploration-fast-001/config/smoke.json', '/research/projects/cellular-language-models/experiments/clm-0.4-mini-language-validation/validation/clm-0.4-mini-language-validation/calibration-assets.json', '/research/projects/cellular-language-models/experiments/clm-0.4-mini-language-validation/validation/clm-0.4-mini-language-validation/protocol.json', '/research/projects/cellular-language-models/experiments/clm-0.4-mini-m1-v2-calibration/artifacts/dev-90402/protocol.json', '/research/projects/cellular-language-models/experiments/clm-0.4-mini-m1-v2-language-validation/validation/clm-0.4-mini-m1-v2-language-validation/protocol.json', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/config/proposed.json', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r2/config/proposed.json', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3/config/proposed.json', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3b/config/proposed.json', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3b/reports/release-review/runtime-candidate.json', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3c/moe-bridge/config/development.json', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3c/moe-bridge/config/pilot.json', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3c/moe-bridge/config/proposed.json', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3d/config/frozen.json', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3e/config/frozen.json', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3f-connection-admission/config/frozen.json', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3f-v2/config/frozen.json', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3i/config/frozen.json', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3k/config/frozen.json', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3l/config/frozen.json', '/research/projects/cellular-language-models/experiments/integrated-replay-free-clm-kill-test-001/artifacts/seed-95301/data-manifest.json', '/research/projects/cellular-language-models/experiments/integrated-replay-free-clm-kill-test-001/artifacts/seed-95301/matched_replay_oracle/arm-summary-core.json', '/research/projects/cellular-language-models/experiments/integrated-replay-free-clm-kill-test-001/artifacts/seed-95301/matched_replay_oracle/arm-summary.json', '/research/projects/cellular-language-models/experiments/native-clm-v0-m2-continual-language/validation/native-clm-v0-m2-continual-language/protocol.json', '/research/projects/cellular-language-models/experiments/native-clm-v0-m3-growth-restored-continual-language/artifacts/data-manifest.json', '/research/projects/cellular-language-models/experiments/native-clm-v0-m3-growth-restored-continual-language/validation/native-clm-v0-m3-growth-restored-continual-language/protocol.json', '/research/projects/cellular-language-models/experiments/native-clm-v0-m3l-query-sketch-gate/artifacts/diagnostic-result.json', '/research/projects/cellular-language-models/experiments/native-clm-v0-m3l1-address-state-capacity/artifacts/diagnostic-result.json', '/research/projects/cellular-language-models/experiments/native-clm-v0-m3l2-online-address-state/artifacts/data-manifest.json', '/research/projects/cellular-language-models/experiments/native-clm-v0-m3l2-online-address-state/validation/native-clm-v0-m3l2-online-address-state/protocol.json', '/research/projects/cellular-language-models/experiments/native-clm-v0-m3r-address-diagnostic/artifacts/diagnostic-result.json', '/research/projects/cellular-language-models/experiments/native-clm-v0-m3r-read-preserving-growth/artifacts/data-manifest.json', '/research/projects/cellular-language-models/experiments/native-clm-v0-m3r-read-preserving-growth/validation/native-clm-v0-m3r-read-preserving-growth/protocol.json', '/research/projects/cellular-computing/experiments/021-conflict-driven-differentiation/artifacts/decision.json', '/research/projects/cellular-computing/experiments/022-emergent-trait-bifurcation/artifacts/decision.json', '/research/projects/cellular-computing/experiments/023b-probationary-trait-genesis/artifacts/decision.json']


def run(command,label=None,timeout=None):
    if label:
        print('[CLM002 Gate0]',label,flush=True)
    return subprocess.run(list(map(str,command)),check=True,timeout=timeout)


if not importlib.util.find_spec('tokenizers'):
    run([sys.executable,'-m','pip','install','--quiet','tokenizers==0.20.3'],'Prepare existing runtime dependency',45)
if not (CODE/'.git').exists():
    CODE.mkdir(parents=True,exist_ok=True)
    run(['git','init','--quiet',CODE])
    run(['git','-C',CODE,'remote','add','origin','https://github.com/ArcheLabs/mini-cells.git'])
    run(['git','-C',CODE,'config','remote.origin.promisor','true'])
    run(['git','-C',CODE,'config','remote.origin.partialclonefilter','blob:none'])
    run(['git','-C',CODE,'sparse-checkout','init','--no-cone'])
    run(['git','-C',CODE,'sparse-checkout','set','--no-cone',*PATTERNS])
    run(['git','-C',CODE,'fetch','--depth=1','--filter=blob:none','origin',SOURCE_SHA],'Fetch pinned source',45)
    run(['git','-C',CODE,'checkout','--detach','FETCH_HEAD'],'Load code and historical metadata',45)
actual=subprocess.check_output(['git','-C',str(CODE),'rev-parse','HEAD'],text=True).strip()
if actual!=SOURCE_SHA:
    raise RuntimeError('CLM002_SOURCE_CHANGED_USE_FRESH_CODE_FOLDER')
entry=CODE/BASE_REL/'run.py'
print('[CLM002 Gate0] Effects: local engineering reports + GitHub results/clm002-gate0-* branch. Optional private parent bytes stay under /tmp, outside Save Version Output.',flush=True)
parent=None
# Existing optional HF_TOKEN: read one pinned ~7.4 MB model object, never a dataset.
token=os.environ.get('HF_TOKEN','').strip()
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token=UserSecretsClient().get_secret('HF_TOKEN').strip()
    except Exception:
        token=''
if token:
    os.environ['HF_TOKEN']=token
    os.environ['HF_HUB_DOWNLOAD_TIMEOUT']='20'
    os.environ['HF_HUB_ETAG_TIMEOUT']='10'
    os.environ['HF_HUB_CACHE']='/tmp/clm002-private-hf-cache'
    if not importlib.util.find_spec('huggingface_hub'):
        run([sys.executable,'-m','pip','install','--quiet','huggingface_hub'],'Prepare private artifact reader',45)
    fetch_code="""import sys,runpy,shutil,json
from pathlib import Path
sys.path.insert(0,str(Path(sys.argv[1]).parent))
try:
 ns=runpy.run_path(sys.argv[1],run_name='clm002_parent_reader');ns['bootstrap']()
 from minicells_clm002.common import configuration,digest
 from minicells_artifacts.hf_storage import HubBackend
 c=configuration();ref=c['substrate']['historical_identity']['hf_reference'];expected=c['substrate']['selected_parent_checkpoint_sha256']
 target=Path(sys.argv[2]);target.parent.mkdir(parents=True,exist_ok=True)
 if target.exists():
  if digest(target)!=expected: raise RuntimeError('CACHED_PARENT_SHA_MISMATCH')
 else:
  original=HubBackend(ref['hf_repo']).get('objects/sha256/'+expected,ref['hf_revision'])
  if digest(original)!=expected: raise RuntimeError('PARENT_SHA_MISMATCH')
  shutil.copyfile(original,target)
 print(json.dumps(dict(status='PINNED_PARENT_BYTES_READY')))
except Exception as error:
 print(json.dumps(dict(status='PARENT_UNAVAILABLE',failure_type=type(error).__name__)))
 sys.exit(1)
"""
    parent_file=Path('/tmp/clm002-parent-inspection/9b2d77efe3d6346fbf46ead8d8e4038af07538813c14125c70c2f36ba48d3ddf.pt')
    try:
        # Deadline bounds this read-only network helper, never training.
        result=subprocess.run([sys.executable,'-c',fetch_code,str(entry),str(parent_file)],capture_output=True,text=True,timeout=60)
        if result.returncode==0:
            parent=parent_file
            print('[CLM002 Gate0] Original parent bytes ready for CPU identity inspection',flush=True)
        else:
            print('[CLM002 Gate0] Parent unavailable; Gate 0 remains BLOCKED. CPU fixtures still carry no scientific claim.',flush=True)
    except subprocess.TimeoutExpired:
        print('[CLM002 Gate0] Parent read timed out; no scientific training started.',flush=True)
else:
    print('[CLM002 Gate0] HF_TOKEN not enabled; parent bytes remain unverified.',flush=True)
command=[sys.executable,entry,'--mode','engineering','--output',OUTPUT,'--publish-github']
if parent is not None:
    command.extend(['--checkpoint',parent])
try:
    run(command,'CPU functional checks, audit and automatic GitHub report return')
finally:
    print('Save Version retains Gate 0 reports and private local checkpoints:',OUTPUT,flush=True)
    print('Scientific GPU training was not authorized or started.',flush=True)
